# Week 6: Neural networks

First you'll build a tiny neural network by hand with NumPy, so you see the maths. Then you'll fine-tune a real image model to spot **cassava mosaic disease** on leaf photos, using photos collected by Makerere University's AI lab in Uganda.

**Before you start:** on Kaggle, set **Accelerator: GPU** and **Internet on** in the settings panel, and add the *Cassava Leaf Disease Classification* competition data with **Add input** (you need to accept the competition rules once).

Run each cell with **Shift + Enter**. Every exercise has a check cell that prints **PASS** or a hint.
At the end you get a completion code for the submission form.

In [ ]:
# Run this cell first. You don't need to read it: it holds the self-checks.
EMAIL = ""  # <- the email you applied with

import hashlib, json, os, re

_passed = set()

def ok(n):
    _passed.add(n)
    print(f"PASS: exercise {n} done.")

def hint(message):
    print(f"Not yet. Hint: {message}")

def free_text(n, text, min_sentences):
    parts = [p for p in re.split(r"[.!?]+", text or "") if len(p.split()) >= 3]
    if len(parts) >= min_sentences:
        print("Saved. A TA will read this answer.")
        ok(n)
    else:
        hint(f"write at least {min_sentences} full sentences inside the quotes.")

def completion_code(week, exercises):
    if "@" not in EMAIL:
        return print("Put your email in the first cell, run it, then run this cell again.")
    missing = sorted(set(exercises) - _passed)
    if missing:
        return print(f"Almost there. Not passed yet: {missing}. Run those exercises and checks again.")
    digest = hashlib.sha256(f"akada-labs-cohort-1|{week}|{EMAIL.strip().lower()}".encode()).hexdigest()
    print(f"Well done! Your completion code is: AK-W{week:02d}-{digest[:6].upper()}")
    print("Paste it into the lab submission form.")

def get_secret(name):
    """Read an API key from Kaggle or Colab secrets. Never paste keys into a cell."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    raise RuntimeError(
        f"No secret called {name}. On Kaggle: Add-ons > Secrets. On Colab: the key icon on the left. "
        "Add it there, switch it on for this notebook, then run this cell again.")

print("Ready.")

## Part 1: a network by hand

A neural network layer multiplies inputs by **weights**, adds a **bias**, and passes the result through an **activation** function.

- Hidden layer: `h = relu(x @ W1 + b1)`, where `relu` turns negatives into 0
- Output layer: `y = sigmoid(h @ W2 + b2)`, which squashes any number to between 0 and 1

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
x = np.array([0.5, -1.2, 3.0, 0.8])      # 4 input numbers
W1 = rng.normal(size=(4, 3)); b1 = np.zeros(3)
W2 = rng.normal(size=(3, 1)); b2 = np.zeros(1)

def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Exercise 1: Forward pass

Compute the hidden layer `h` and the output `y` using the formulas above.

In [ ]:
h = None
y = None

print(h, y)

In [ ]:
try:
    want_h = relu(x @ W1 + b1)
    want_y = sigmoid(want_h @ W2 + b2)
    if np.shape(y) != (1,):
        hint("y should have shape (1,). Use h @ W2 + b2 inside sigmoid.")
    elif not (np.allclose(h, want_h) and np.allclose(y, want_y)):
        hint("check the order: x @ W1 + b1, then relu, then @ W2 + b2, then sigmoid.")
    else:
        ok(1)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Learning = reducing the loss

Say the right answer for `x` is `1` (diseased). The **loss** measures how wrong `y` is:
`loss = -log(y)` (lower is better). Training nudges the weights a little in the direction that lowers the loss.
For the output weights, that direction (the **gradient**) is `h * (y - 1)`, reshaped to match `W2`.

## Exercise 2: One gradient step

Compute `loss_before`. Update `W2_new = W2 - 0.1 * grad`. Then recompute the output with `W2_new` and store `loss_after`.

In [ ]:
target = 1.0
loss_before = None
grad = (h * (y - target)).reshape(3, 1)
W2_new = None
loss_after = None

print(loss_before, loss_after)

In [ ]:
try:
    want_before = float(-np.log(sigmoid(relu(x @ W1 + b1) @ W2 + b2))[0])
    if not np.isclose(float(np.squeeze(loss_before)), want_before):
        hint("loss_before = -np.log(y)")
    elif float(np.squeeze(loss_after)) >= float(np.squeeze(loss_before)):
        hint("the loss should go down. Did you subtract 0.1 * grad and recompute y with W2_new?")
    else:
        print(f"Loss went from {float(np.squeeze(loss_before)):.3f} to {float(np.squeeze(loss_after)):.3f}")
        ok(2)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Part 2: a real image model

We keep two classes: **healthy** leaves (label 4 in the competition data) and **cassava mosaic disease** (label 3),
about 500 photos each. Rather than train from zero, we start from **ResNet-18**, a model already trained on millions of photos,
and fine-tune it. That's called **transfer learning**.

In [ ]:
import pandas as pd, torch, torchvision
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

DATA = Path("/kaggle/input/cassava-leaf-disease-classification")
CLASSES = {4: "healthy", 3: "mosaic disease"}
PER_CLASS = 500

labels = pd.read_csv(DATA / "train.csv")
subset = pd.concat([labels[labels["label"] == k].sample(PER_CLASS, random_state=42) for k in CLASSES])
subset["target"] = (subset["label"] == 3).astype(int)   # 1 = diseased
subset = subset.sample(frac=1, random_state=42).reset_index(drop=True)
train_df, test_df = subset.iloc[:800], subset.iloc[800:]

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class Leaves(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = Image.open(DATA / "train_images" / row["image_id"]).convert("RGB")
        return tfm(image), int(row["target"])

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device, "| train:", len(train_df), "test:", len(test_df))

## Exercise 3: Look at a batch

Make `train_loader = DataLoader(Leaves(train_df), batch_size=32, shuffle=True)` and the same for `test_loader` (with `shuffle=False`). Take one batch with `images, targets = next(iter(train_loader))` and show the first 8 images with their labels.

In [ ]:
train_loader = None
test_loader = None
images, targets = None, None

# Showing images: undo the normalisation so colours look right
# fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
# for ax, img, t in zip(axes, images, targets):
#     ax.imshow((img.permute(1, 2, 0) * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])).clamp(0, 1))
#     ax.set_title(["healthy", "diseased"][t]); ax.axis("off")

In [ ]:
try:
    if len(train_loader.dataset) != 800 or len(test_loader.dataset) != 200:
        hint("use Leaves(train_df) and Leaves(test_df).")
    elif tuple(images.shape[1:]) != (3, 224, 224) or len(images) != len(targets):
        hint("images should come from next(iter(train_loader)).")
    else:
        ok(3)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, targets in loader:
            preds = model(images.to(device)).argmax(1).cpu()
            correct += (preds == targets).sum().item()
            total += len(targets)
    return correct / total

## Exercise 4: Fine-tune for 2 epochs

Load a pretrained ResNet-18, swap its last layer for one with 2 outputs, and train for 2 epochs. Fill in the 4 lines marked `TODO`. One epoch is one pass over the training photos. Aim for test accuracy of at least 75%.

In [ ]:
model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
model.fc = torch.nn.Linear(model.fc.in_features, 2)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(2):
    model.train()
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = None    # TODO: run the model on images
        loss = None       # TODO: loss_fn(outputs, targets)
        # TODO: loss.backward()
        # TODO: optimizer.step()
    print(f"epoch {epoch + 1}: test accuracy {evaluate(model, test_loader):.0%}")

test_accuracy = evaluate(model, test_loader)

In [ ]:
try:
    if test_accuracy < 0.75:
        hint(f"test accuracy is {test_accuracy:.0%}. Check all 4 TODO lines, then train again.")
    else:
        ok(4)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 5: Where it goes wrong

The code shows up to 5 test photos the model got wrong. Look closely. In `answer`, write 2 or more sentences: why do you think it missed them? (Blurry? Shadows? Disease only on part of the leaf?)

In [ ]:
model.eval()
wrong = []
test_set = Leaves(test_df)
with torch.no_grad():
    for i in range(len(test_set)):
        image, target = test_set[i]
        pred = model(image.unsqueeze(0).to(device)).argmax(1).item()
        if pred != target:
            wrong.append((test_df.iloc[i]["image_id"], target, pred))
print(len(wrong), "wrong out of", len(test_df))

fig, axes = plt.subplots(1, max(1, min(5, len(wrong))), figsize=(16, 3.5), squeeze=False)
for ax, (image_id, target, pred) in zip(axes[0], wrong[:5]):
    ax.imshow(Image.open(DATA / "train_images" / image_id))
    ax.set_title(f"real: {['healthy', 'diseased'][target]}\nguess: {['healthy', 'diseased'][pred]}")
    ax.axis("off")
plt.show()

answer = """

"""

In [ ]:
try:
    free_text(5, answer, 2)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Finish

Run the cell below. If every exercise passed, it prints your completion code.

In [ ]:
completion_code(6, [1, 2, 3, 4, 5])